In [1]:
# Upload your LAB-Project-SB.zip file
from google.colab import files
import zipfile
import os

print('Upload LAB-Project-SB.zip:')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

# Extract
with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall('.')

# Find the extracted folder
project_dir = 'LAB-Project-SB'
if os.path.exists(project_dir):
    print(f'✓ Extracted to: {project_dir}')
    print(f'  Files: {os.listdir(project_dir)[:10]}')
else:
    print(' Folder not found!')


Upload LAB-Project-SB.zip:


Saving LAB-Project-SB.zip to LAB-Project-SB.zip
✓ Extracted to: LAB-Project-SB
  Files: ['model.py', 'finetune_real_fixednorm_best.pt', 'finetune_real_best.pt', 'Sneha_Basker_NN_GARCH_VS_GARCH_FINAL.ipynb', 'Main.ipynb', 'finetune_real_zeroaware_r2min1e8_best.pt', 'Portfolio.ipynb', '__pycache__', 'sigma_vs_absr_diagnostic.png', 'portfolio_comparison.png']


In [2]:
from __future__ import annotations
import warnings, sys, time
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import cvxpy as cp

warnings.filterwarnings('ignore')
sys.path.insert(0, 'LAB-Project-SB')  # Add to path

# Import from your files
from model import HybridGarch
from data_stocks_WIKI_price import get_cleaned_data

print(f'PyTorch {torch.__version__} | NumPy {np.__version__} | cvxpy {cp.__version__}')
print('✓ Imported from LAB-Project-SB')


PyTorch 2.10.0+cpu | NumPy 2.0.2 | cvxpy 1.6.7
✓ Imported from LAB-Project-SB


In [3]:
!pip install rienet-torch


In [4]:
# Configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CHECKPOINT = 'LAB-Project-SB/finetune_real_zeroaware_r2min1e8_best.pt'

WINDOW_SIZE = 90
N_STOCKS = 30
TRAIN_RATIO = 0.70
ANNUALIZE = 252
MAX_WEIGHT = 0.10

print(f'Device: {DEVICE} | Window: {WINDOW_SIZE} | Max weight: {MAX_WEIGHT:.0%}')


Device: cpu | Window: 90 | Max weight: 10%


In [5]:
# ═══════════════════════════════════════════════════════════════
#  DATA LOADING: Google Drive (no 50 min upload!)
# ═══════════════════════════════════════════════════════════════

from google.colab import drive

# Mount Google Drive
print('Mounting Google Drive...')
drive.mount('/content/drive')
print('✓ Google Drive mounted')

# Find WIKI_PRICES file
import os

print('\nSearching for WIKI_PRICES file...')
found_files = []
search_paths = [
    '/content/drive/MyDrive/',
    '/content/drive/MyDrive/Colab Notebooks/',
]

for path in search_paths:
    if os.path.exists(path):
        for root, dirs, files in os.walk(path):
            for f in files:
                if 'WIKI_PRICES' in f and f.endswith('.zip'):
                    full_path = os.path.join(root, f)
                    found_files.append(full_path)
                    print(f'✓ FOUND: {full_path}')

if not found_files:
    print(' WIKI_PRICES not found in Google Drive!')
    print('\n Please:')
    print('   1. Upload WIKI_PRICES_*.zip to your Google Drive')
    print('   2. Rerun this cell')
    raise FileNotFoundError('WIKI_PRICES_*.zip not found')

# Use the first found file
FILE_PATH = found_files[0]
print(f'\n✓ Using: {FILE_PATH}')

# Or manually set the path if you know it:
# FILE_PATH = '/content/drive/MyDrive/WIKI_PRICES_212b326a081eacca455e13140d7bb9db.zip'


Mounting Google Drive...
Mounted at /content/drive
✓ Google Drive mounted

Searching for WIKI_PRICES file...
✓ FOUND: /content/drive/MyDrive/WIKI_PRICES_212b326a081eacca455e13140d7bb9db.zip

✓ Using: /content/drive/MyDrive/WIKI_PRICES_212b326a081eacca455e13140d7bb9db.zip


In [6]:
# ═══════════════════════════════════════════════════════════════
#  DATA LOADING FUNCTION
# ═══════════════════════════════════════════════════════════════

import time

def load_from_2008(filepath):
    """
    Load WIKI prices from 2008 onwards.
    Removes dropna() bug - keeps all data!
    """
    print('\n Loading from 2008...')
    t0 = time.time()

    with zipfile.ZipFile(filepath) as z:
        with z.open(z.namelist()[0]) as f:
            df = pd.read_csv(f, parse_dates=['date'], usecols=['date', 'ticker', 'adj_close'])

    print(f'Read {len(df):,} rows in {time.time()-t0:.0f}s')

    # Filter to 2008+
    df = df[df['date'] >= '2008-01-01']
    print(f'After 2008 filter: {len(df):,} rows')

    # Pivot
    prices = df.pivot(index='date', columns='ticker', values='adj_close')
    print(f'After pivot: {prices.shape[0]} days × {prices.shape[1]} stocks')

    # Compute returns (NO dropna!)
    returns = prices.pct_change()
    returns = returns.iloc[1:]  # Remove only first row (NaN from pct_change)

    print(f'After pct_change: {returns.shape[0]} days × {returns.shape[1]} stocks')

    # Keep stocks with ≥30% valid data
    valid = returns.notna().mean()
    good = valid[valid >= 0.30].index
    returns = returns[good]

    # Fill remaining NaN with 0
    returns = returns.fillna(0)

    print(f'Final: {len(returns)} days × {len(good)} stocks')
    print(f'Time: {time.time()-t0:.0f}s')

    return returns

# Load data
returns_df = load_from_2008(FILE_PATH)

print(f'\n✓ Data loaded: {returns_df.shape}')



⚡ Loading from 2008...
Read 15,389,314 rows in 28s
After 2008 filter: 7,126,319 rows
After pivot: 2635 days × 3199 stocks
After pct_change: 2634 days × 3199 stocks
Final: 2634 days × 3164 stocks
Time: 32s

✓ Data loaded: (2634, 3164)


In [7]:
# ═══════════════════════════════════════════════════════════════
#  DATA PROCESSING: Select stocks FIRST, then split
# ═══════════════════════════════════════════════════════════════

# 1. SELECT STOCKS FIRST!
returns_df = returns_df[returns_df.columns[:N_STOCKS]]
print(f'✓ Selected {N_STOCKS} stocks: {list(returns_df.columns[:5])}...')

# 2. Train/test split
n_total = len(returns_df)
split_idx = int(TRAIN_RATIO * n_total)
start_t = max(WINDOW_SIZE, split_idx)

returns_train = returns_df.iloc[:split_idx]
returns_test = returns_df.iloc[split_idx:]
test_dates = returns_df.index[start_t:]

# 3. Convert to array and compute statistics
returns_arr = returns_df.values.astype(np.float32)
train_means = returns_train.mean().values.astype(np.float32)
train_stds = returns_train.std(ddof=0).values.astype(np.float32)

print(f'\n✓ Data shapes:')
print(f'  returns_df: {returns_df.shape}')
print(f'  returns_arr: {returns_arr.shape}')
print(f'  train_means: {train_means.shape}')
print(f'  train_stds: {train_stds.shape}')

print(f'\n✓ Split:')
print(f'  Train: {returns_train.index[0].date()} → {returns_train.index[-1].date()} ({len(returns_train)} days)')
print(f'  Test:  {returns_test.index[0].date()} → {returns_test.index[-1].date()} ({len(returns_test)} days)')
print(f'  Start_t: {start_t} | Test length: {len(test_dates)}')


✓ Selected 30 stocks: ['A', 'AAL', 'AAMC', 'AAN', 'AAOI']...

✓ Data shapes:
  returns_df: (2634, 30)
  returns_arr: (2634, 30)
  train_means: (30,)
  train_stds: (30,)

✓ Split:
  Train: 2008-01-02 → 2015-02-04 (1843 days)
  Test:  2015-02-05 → 2018-03-27 (791 days)
  Start_t: 1843 | Test length: 791


In [8]:
# Load trained model
model = HybridGarch(hidden=32, kernel=3, conv_layers=5)
model.load_state_dict(torch.load(CHECKPOINT, map_location=DEVICE))
model.to(DEVICE)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f'✓ Model loaded: {n_params:,} parameters')


✓ Model loaded: 14,499 parameters


In [9]:
@torch.no_grad()
def precompute_sigma2_nn(returns_arr, train_means, train_stds, model, device, window_size, start_t, annualize=252):
    T, N = returns_arr.shape
    sigma2_mat = np.full((T - start_t, N), np.nan)
    eps = 1e-8

    for k, t in enumerate(range(start_t, T)):
        windows = returns_arr[t - window_size:t, :]
        if np.isnan(windows).any():
            continue
        wn = (windows.T - train_means[:, None]) / (train_stds[:, None] + eps)
        x = torch.tensor(wn, dtype=torch.float32, device=device).unsqueeze(-1)
        s2, _, _, _ = model(x)
        sigma2_mat[k] = np.clip(s2.cpu().numpy() * (train_stds + eps) ** 2, 1e-10, None) * annualize

    return sigma2_mat


def precompute_sigma2_arch(returns_arr, split_idx, start_t, annualize=252):
    """Simple GARCH(1,1) with scipy.optimize"""
    from scipy.optimize import minimize

    T, N = returns_arr.shape
    sigma2_mat = np.zeros((T - start_t, N))

    for j in range(N):
        r_j = returns_arr[:split_idx, j]
        r_j = r_j[~np.isnan(r_j)]

        if len(r_j) < 30:
            sigma2_mat[:, j] = np.var(r_j) * annualize if len(r_j) > 0 else 0.01 * annualize
            continue

        try:
            uvar = np.var(r_j)

            def nll(p):
                o, a, b = p
                if o <= 0 or a < 0 or b < 0 or a + b >= 1:
                    return 1e10
                s2 = np.full(len(r_j), uvar)
                for t in range(1, len(r_j)):
                    s2[t] = o + a * r_j[t-1]**2 + b * s2[t-1]
                    if s2[t] <= 0:
                        return 1e10
                try:
                    return 0.5 * np.sum(np.log(2*np.pi) + np.log(s2) + r_j**2/s2)
                except:
                    return 1e10

            res = minimize(nll, [uvar*0.01, 0.05, 0.90], method='L-BFGS-B',
                          bounds=[(1e-8, uvar), (1e-4, 0.5), (0.3, 0.99)])
            omega, alpha, beta = res.x
        except:
            omega, alpha, beta = 1e-6, 0.05, 0.90

        s2 = omega / (1 - alpha - beta + 1e-8)
        for t in range(split_idx, T):
            r_prev = returns_arr[t-1, j] if not np.isnan(returns_arr[t-1, j]) else 0.0
            s2 = omega + alpha * r_prev**2 + beta * s2
            s2 = max(s2, 1e-10)
            if t >= start_t:
                sigma2_mat[t - start_t, j] = s2 * annualize

    return sigma2_mat


print('✓ Volatility functions ready')


✓ Volatility functions ready


In [10]:
print('[1/2] CNN-GARCH forward pass...')
t0 = time.time()
sigma2_nn = precompute_sigma2_nn(returns_arr, train_means, train_stds, model, DEVICE, WINDOW_SIZE, start_t, ANNUALIZE)
print(f'  ✓ Done ({time.time()-t0:.1f}s)')

print('[2/2] GARCH(1,1) recursion...')
t0 = time.time()
sigma2_arch = precompute_sigma2_arch(returns_arr, split_idx, start_t, ANNUALIZE)
print(f'  ✓ Done ({time.time()-t0:.1f}s)')

print(f'\nShape: {sigma2_nn.shape} [test_days × N_stocks]')


[1/2] CNN-GARCH forward pass...
  ✓ Done (5.1s)
[2/2] GARCH(1,1) recursion...
  ✓ Done (13.6s)

Shape: (791, 30) [test_days × N_stocks]


In [11]:
def rolling_correlation(returns_arr, t, window_size):
    window = returns_arr[t - window_size:t, :]
    w = window - window.mean(axis=0)
    cov = (w.T @ w) / (window_size - 1)
    std = np.maximum(np.sqrt(np.diag(cov)), 1e-10)
    corr = cov / np.outer(std, std)
    corr = np.clip(corr, -1.0, 1.0)
    np.fill_diagonal(corr, 1.0)
    return corr

def nearest_psd(cov, reg=1e-6):
    cov = (cov + cov.T) / 2
    eigvals, eigvecs = np.linalg.eigh(cov)
    eigvals = np.maximum(eigvals, reg)
    cov_psd = eigvecs @ np.diag(eigvals) @ eigvecs.T
    return (cov_psd + cov_psd.T) / 2

def build_covariance(sigma2_daily, corr):
    sigma_daily = np.sqrt(np.maximum(sigma2_daily / ANNUALIZE, 1e-10))
    D = np.diag(sigma_daily)
    return nearest_psd(D @ corr @ D)

def gmv_weights(cov, max_weight=0.10):
    """FIXED: Cholesky trick (no PSD check!)"""
    N = cov.shape[0]
    cov_psd = nearest_psd(cov, reg=1e-6)

    try:
        L = np.linalg.cholesky(cov_psd)
    except:
        return np.ones(N) / N

    w = cp.Variable(N)
    objective = cp.Minimize(cp.sum_squares(L.T @ w))  # ← FIXED!
    constraints = [cp.sum(w) == 1, w >= 0, w <= max_weight]
    prob = cp.Problem(objective, constraints)

    try:
        prob.solve(solver=cp.CLARABEL, verbose=False)
    except:
        try:
            prob.solve(solver=cp.ECOS, verbose=False)
        except:
            return np.ones(N) / N

    if prob.status in ('optimal', 'optimal_inaccurate') and w.value is not None:
        weights = np.maximum(np.array(w.value).flatten(), 0)
        s = weights.sum()
        return weights / s if s > 1e-10 else np.ones(N) / N
    return np.ones(N) / N

print('✓ GMV functions (Cholesky - FIXED!)')


# ═══════════════════════════════════════════════════════════════
#  RIENET: Complete Implementation with Inverse Eigenvalues
# ═══════════════════════════════════════════════════════════════

from rienet_torch import CorrelationEigenTransformLayer
import torch

# Initialize RIEnet
rie_layer = CorrelationEigenTransformLayer(name="portfolio_corr_cleaner")

def rolling_correlation_RIENET_FULL(returns_arr, t, window_size):
    """
    Complete RIEnet with ALL outputs:
    - correlation (cleaned)
    - inverse_correlation
    - eigenvalues
    - inverse_eigenvalues  ← Professor wants this!
    - eigenvectors
    """
    window = returns_arr[t - window_size:t, :]

    # Compute sample correlation
    w = window - window.mean(axis=0)
    cov = (w.T @ w) / (window_size - 1)
    std = np.maximum(np.sqrt(np.diag(cov)), 1e-10)
    sample_corr = cov / np.outer(std, std)
    sample_corr = np.clip(sample_corr, -1.0, 1.0)
    np.fill_diagonal(sample_corr, 1.0)

    # Convert to tensor
    corr_tensor = torch.tensor(sample_corr, dtype=torch.float32).unsqueeze(0)

    # Attributes: [volatility_regime, lookback]
    vol_regime = window.std()
    attrs = torch.tensor([[vol_regime, float(window_size)]], dtype=torch.float32)

    # Get ALL outputs from RIEnet
    with torch.no_grad():
        details = rie_layer(
            corr_tensor,
            attributes=attrs,
            output_type=[
                "correlation",
                "inverse_correlation",
                "eigenvalues",
                "inverse_eigenvalues",  # ← Key for stable GMV!
                "eigenvectors"
            ]
        )

    # Extract all components
    cleaned_corr = details["correlation"].squeeze(0).numpy()
    inv_corr = details["inverse_correlation"].squeeze(0).numpy()
    eigvals = details["eigenvalues"].squeeze(0).numpy().flatten()  # (n_assets,)
    inv_eigvals = details["inverse_eigenvalues"].squeeze(0).numpy().flatten()  # (n_assets,)
    eigvecs = details["eigenvectors"].squeeze(0).numpy()  # (n_assets, n_assets)

    return {
        "correlation": cleaned_corr,
        "inverse_correlation": inv_corr,
        "eigenvalues": eigvals,
        "inverse_eigenvalues": inv_eigvals,
        "eigenvectors": eigvecs
    }

def gmv_weights_with_eigen(eigvecs, inv_eigvals, max_weight=0.10):
    """
    GMV using inverse eigenvalues with constraints.

    Steps:
    1. Try unconstrained GMV formula
    2. If constraints violated, solve with cvxpy
    """
    N = eigvecs.shape[0]
    ones = np.ones(N)

    # STEP 1: Unconstrained GMV using inverse eigenvalues
    # Formula: w = Σ^-1 @ 1 / (1^T @ Σ^-1 @ 1)
    # Where:   Σ^-1 = V @ Λ^-1 @ V^T

    temp = eigvecs.T @ ones           # V^T @ 1
    temp = inv_eigvals * temp         # Λ^-1 @ (V^T @ 1)
    inv_cov_ones = eigvecs @ temp     # V @ (Λ^-1 @ V^T @ 1)

    denom = ones @ inv_cov_ones

    if abs(denom) < 1e-10:
        return np.ones(N) / N  # Singular case

    w_unconstrained = inv_cov_ones / denom

    # STEP 2: Check if constraints are satisfied
    if (w_unconstrained >= -1e-6).all() and (w_unconstrained <= max_weight + 1e-6).all():
        # Constraints satisfied! Use analytical solution
        w_clean = np.maximum(w_unconstrained, 0)  # Remove tiny negative values
        return w_clean / w_clean.sum()

    # STEP 3: Constraints violated - solve with cvxpy
    # Reconstruct covariance from eigendecomposition
    Lambda = np.diag(1.0 / np.maximum(inv_eigvals, 1e-10))
    cov = eigvecs @ Lambda @ eigvecs.T
    cov = nearest_psd(cov)  # Ensure PSD

    # Solve constrained optimization
    w = cp.Variable(N)
    objective = cp.Minimize(cp.quad_form(w, cov))
    constraints = [
        cp.sum(w) == 1,
        w >= 0,
        w <= max_weight
    ]
    prob = cp.Problem(objective, constraints)

    try:
        prob.solve(solver=cp.CLARABEL, verbose=False)
        if prob.status in ('optimal', 'optimal_inaccurate'):
            wgt = np.maximum(w.value.flatten(), 0)
            return wgt / wgt.sum()
    except:
        pass

    # Fallback to equal weights
    return np.ones(N) / N

print('✓ GMV functions + RIEnet (COMPLETE with inverse eigenvalues!)')


✓ GMV functions (Cholesky - FIXED!)
✓ GMV functions + RIEnet (COMPLETE with inverse eigenvalues!)


In [12]:
def compute_gmv_portfolio(sigma2_mat, returns_arr, start_t, window_size, max_weight, label):
    test_len, N = sigma2_mat.shape
    port_returns = np.zeros(test_len)
    weights_hist = np.zeros((test_len, N))

    for k in range(test_len):
        t = start_t + k
        r_t = returns_arr[t]
        s2_k = sigma2_mat[k]

        if np.isnan(s2_k).any():
            w = np.ones(N) / N
        else:
            corr = rolling_correlation(returns_arr, t, window_size)
            cov = build_covariance(s2_k, corr)
            w = gmv_weights(cov, max_weight=max_weight)

        port_returns[k] = np.dot(w, r_t)
        weights_hist[k] = w

        if k % 200 == 0:
            print(f'  [{label}] {k}/{test_len} — max_w={w.max():.1%} HHI={(w**2).sum():.3f}')

    return port_returns, weights_hist

def inverse_variance_weights(sigma2, eps=1e-10):
    s2 = sigma2.copy()
    nm = np.isnan(s2) | (s2 <= 0)
    if nm.all():
        return np.ones(len(s2)) / len(s2)
    s2[nm] = np.nanmean(s2[~nm])
    inv = 1.0 / np.maximum(s2, eps)
    return inv / inv.sum()

def compute_inv_var_portfolio(sigma2_mat, returns_arr, start_t):
    test_len, N = sigma2_mat.shape
    port_returns = np.zeros(test_len)
    weights_hist = np.zeros((test_len, N))
    for k in range(test_len):
        w = inverse_variance_weights(sigma2_mat[k])
        port_returns[k] = np.dot(w, returns_arr[start_t + k])
        weights_hist[k] = w
    return port_returns, weights_hist

print('✓ Portfolio functions ready')


✓ Portfolio functions ready


In [13]:
print('Computing all strategies...\n')

# 1. Equal-Weight
N = N_STOCKS
test_len = len(returns_arr) - start_t
r_eq = np.array([np.dot(np.ones(N)/N, returns_arr[t]) for t in range(start_t, len(returns_arr))])
w_eq = np.tile(np.ones(N)/N, (len(r_eq), 1))
print('[1/5] Equal-Weight ✓')

# 2. Inv-Var NN
r_inv_nn = np.zeros(test_len)
for k in range(test_len):
    w = inverse_variance_weights(sigma2_nn[k])
    r_inv_nn[k] = np.dot(w, returns_arr[start_t + k])
print('[2/5] Inv-Var NN ✓')

# 3. GMV-GARCH
print('[3/5] GMV-GARCH...')
r_gmv_arch, w_gmv_arch = compute_gmv_portfolio(
    sigma2_arch, returns_arr, start_t, WINDOW_SIZE, MAX_WEIGHT, 'GMV-GARCH'
)
print('  ✓ Done')

# 4. GMV-NN
print('[4/5] GMV-NN...')
r_gmv_nn, w_gmv_nn = compute_gmv_portfolio(
    sigma2_nn, returns_arr, start_t, WINDOW_SIZE, MAX_WEIGHT, 'GMV-NN'
)
print('  ✓ Done')

print('\n All strategies computed!')


Computing all strategies...

[1/5] Equal-Weight ✓
[2/5] Inv-Var NN ✓
[3/5] GMV-GARCH...
  [GMV-GARCH] 0/791 — max_w=10.0% HHI=0.088
  [GMV-GARCH] 200/791 — max_w=10.0% HHI=0.091
  [GMV-GARCH] 400/791 — max_w=10.0% HHI=0.088
  [GMV-GARCH] 600/791 — max_w=10.0% HHI=0.084
  ✓ Done
[4/5] GMV-NN...
  [GMV-NN] 0/791 — max_w=10.0% HHI=0.088
  [GMV-NN] 200/791 — max_w=10.0% HHI=0.088
  [GMV-NN] 400/791 — max_w=10.0% HHI=0.090
  [GMV-NN] 600/791 — max_w=10.0% HHI=0.088
  ✓ Done

✅ All strategies computed!


In [14]:
print('[5/5] GMV-RIEnet (using RIEnet cleaned correlation)...')

# Define all needed variables
test_len = len(returns_arr) - start_t
window_size = WINDOW_SIZE
max_weight = MAX_WEIGHT
annualize_factor = ANNUALIZE
N = N_STOCKS

r_rienet, w_rienet = [], []
for k in range(test_len):
    t = start_t + k
    s2_k = sigma2_nn[k]

    if np.isnan(s2_k).any():
        w = np.ones(N) / N
    else:
        # Get RIEnet cleaned correlation
        rie_outputs = rolling_correlation_RIENET_FULL(returns_arr, t, window_size)

        # Build covariance using CLEANED correlation
        sigma_daily = np.sqrt(np.maximum(s2_k / annualize_factor, 1e-10))
        D = np.diag(sigma_daily)
        cleaned_corr = rie_outputs["correlation"]
        cov = D @ cleaned_corr @ D

        # Use REGULAR gmv_weights function (same as GMV-NN and GMV-GARCH)
        w = gmv_weights(nearest_psd(cov), max_weight=max_weight)

    r_rienet.append(np.dot(w, returns_arr[t]))
    w_rienet.append(w)

    if k % 200 == 0:
        hhi_k = np.sum(w**2)
        print(f'  [GMV-RIEnet] {k}/{test_len} — max_w={max_weight*100:.1f}% HHI={hhi_k:.3f}')

r_gmv_rienet = np.array(r_rienet)
print('  ✓ Done')


[5/5] GMV-RIEnet (using RIEnet cleaned correlation)...
  [GMV-RIEnet] 0/791 — max_w=10.0% HHI=0.067
  [GMV-RIEnet] 200/791 — max_w=10.0% HHI=0.062
  [GMV-RIEnet] 400/791 — max_w=10.0% HHI=0.061
  [GMV-RIEnet] 600/791 — max_w=10.0% HHI=0.068
  ✓ Done


In [15]:
def financial_metrics(returns, annualize=252):
    r = pd.Series(returns).dropna()
    ann_ret = r.mean() * annualize
    ann_vol = r.std(ddof=0) * np.sqrt(annualize)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else 0
    cumret = (1 + r).cumprod()
    rolling_max = cumret.cummax()
    dd = (cumret - rolling_max) / rolling_max
    max_dd = float(dd.min())
    calmar = ann_ret / abs(max_dd) if max_dd < 0 else 0
    return {
        'Ann. Return': ann_ret,
        'Ann. Volatility': ann_vol,
        'Sharpe Ratio': sharpe,
        'Max Drawdown': max_dd,
        'Calmar Ratio': calmar
    }

# Compute metrics for ALL 5 strategies
m_eq = financial_metrics(r_eq)
m_inv = financial_metrics(r_inv_nn)
m_garch = financial_metrics(r_gmv_arch)
m_nn = financial_metrics(r_gmv_nn)
m_rienet = financial_metrics(r_gmv_rienet)  # ← ADDED!

# Display ALL 5 strategies
print('='*90)
print('PORTFOLIO COMPARISON - ALL 5 STRATEGIES')
print('='*90)
print(f'{"Metric":<20} {"Equal-Wt":>12} {"Inv-Var NN":>12} {"GMV-GARCH":>12} {"GMV-NN":>12} {"GMV-RIEnet":>12}')
print('-'*90)
for k in m_eq:
    print(f'{k:<20} {m_eq[k]:>12.4f} {m_inv[k]:>12.4f} {m_garch[k]:>12.4f} {m_nn[k]:>12.4f} {m_rienet[k]:>12.4f}')

print('\n' + '='*90)
print('GMV-RIENET vs GMV-NN (KEY COMPARISON!)')
print('='*90)
print(f'{"Metric":<20} {"GMV-NN":>18} {"GMV-RIEnet":>18} {"Improvement":>20}')
print('-'*90)
for k in m_nn:
    v1, v2 = m_nn[k], m_rienet[k]
    if 'Vol' in k or 'Drawdown' in k:
        pct = (v1-v2)/abs(v1)*100 if v1 != 0 else 0
        imp = f'{pct:+.1f}% ↓' if v2 < v1 else f'{pct:+.1f}%'
    else:
        pct = (v2-v1)/abs(v1)*100 if v1 != 0 else 0
        imp = f'{pct:+.1f}% ↑' if v2 > v1 else f'{pct:+.1f}%'
    print(f'{k:<20} {v1:>18.4f} {v2:>18.4f} {imp:>24}')

sharpe_imp = (m_rienet['Sharpe Ratio'] - m_nn['Sharpe Ratio']) / m_nn['Sharpe Ratio'] * 100
print(f'\n RIEnet Sharpe improvement over GMV-NN: {sharpe_imp:+.1f}%')



PORTFOLIO COMPARISON - ALL 5 STRATEGIES
Metric                   Equal-Wt   Inv-Var NN    GMV-GARCH       GMV-NN   GMV-RIEnet
------------------------------------------------------------------------------------------
Ann. Return                0.1073       0.0196       0.0443       0.0286       0.0514
Ann. Volatility            0.1443       0.0442       0.0799       0.0761       0.0801
Sharpe Ratio               0.7435       0.4428       0.5540       0.3764       0.6420
Max Drawdown              -0.2326      -0.0748      -0.1161      -0.1140      -0.1270
Calmar Ratio               0.4614       0.2614       0.3813       0.2513       0.4049

GMV-RIENET vs GMV-NN (KEY COMPARISON!)
Metric                           GMV-NN         GMV-RIEnet          Improvement
------------------------------------------------------------------------------------------
Ann. Return                      0.0286             0.0514                 +79.6% ↑
Ann. Volatility                  0.0761             0.0801